# 04 — Evaluate + error analysis (PackSure)

Runs ONLY on the held-out test split (never used for tuning). Reports:
per-entity P/R/F1, exact entity match, field-level accuracy, missing-field
detection, OCR-noise robustness, and inference latency.

> Do not report any accuracy that was not produced by this notebook (or its equivalent).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "transformers>=4.48" datasets seqeval torch

MODEL_DIR = '/content/drive/MyDrive/packsure/ml/runs/YOUR_RUN/export'  # the exported artifact
TEST_JSONL = '/content/drive/MyDrive/packsure/ml/dataset/test/test.jsonl'

In [ ]:
# Load the trained model + test split.
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments

model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
label_list = model.config.id2label
ds = load_dataset('json', data_files={'test': TEST_JSONL})['test']
print('test samples:', len(ds))

In [ ]:
# Tokenize + predict on the test split.
def tokenize(batch):
    tokenized = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=192)
    return tokenized
encoded = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

args = TrainingArguments(output_dir='/tmp/eval', per_device_eval_batch_size=32, report_to=[])
trainer = Trainer(model=model, args=args, data_collator=DataCollatorForTokenClassification(tokenizer))
raw = trainer.predict(encoded)
import numpy as np
pred_ids = np.argmax(raw.predictions, axis=-1)

In [ ]:
# Per-entity P/R/F1 + exact-match + field-level accuracy.
from seqeval.metrics import classification_report, f1_score

def decode(i):
    word_ids = encoded[i].word_ids if False else None
    return None

true_seqs, pred_seqs = [], []
entity_exact = total_entities = exact_hits = 0
for i in range(len(ds)):
    labels = ds[i]['labels']
    word_ids = tokenizer(ds[i]['tokens'], is_split_into_words=True, truncation=True, max_length=192).word_ids()
    pred_row, true_row = [], []
    seen = set()
    for j, w in enumerate(word_ids):
        if w is None or w in seen:
            continue
        seen.add(w)
        pred_row.append(label_list[int(pred_ids[i][j])])
        true_row.append(labels[w])
    true_seqs.append(true_row); pred_seqs.append(pred_row)
    # exact entity match (type + token span)
    def spans(seq):
        out, start = [], None
        for k, lab in enumerate(seq + ['O']):
            if lab.startswith('B-') or lab == 'O':
                if start is not None:
                    out.append((start, k, active))
                start, active = (k, lab[2:]) if lab != 'O' else (None, None)
            elif lab.startswith('I-') and start is None:
                start, active = k, lab[2:]
        return out
    t_spans, p_spans = spans(true_row), spans(pred_row)
    total_entities += len(t_spans)
    exact_hits += len(set(t_spans) & set(p_spans))

print(classification_report(true_seqs, pred_seqs, digits=3))
print(f'exact entity match: {exact_hits}/{total_entities} = {exact_hits / max(total_entities, 1):.3f}')

In [ ]:
# Field-level accuracy: does each FIELD end up with the right full value?
FIELD_OF = lambda label: label.split('-', 1)[1] if label != 'O' else None

def entity_values(tokens, seq):
    values, current, ctype = {}, None, None
    for token, label in zip(tokens, seq):
        if label.startswith('B-'):
            current, ctype = token, label[2:]
        elif label.startswith('I-') and ctype:
            current += ' ' + token
        elif label == 'O':
            current, ctype = None, None
            continue
        if current is not None:
            values[ctype] = current.strip()
        if label == 'O':
            current, ctype = None, None
    return values

field_hits = field_total = 0
missed_fields = []
for i in range(len(ds)):
    gold = entity_values(ds[i]['tokens'], true_seqs[i])
    got = entity_values(ds[i]['tokens'], pred_seqs[i])
    for field, value in gold.items():
        field_total += 1
        if got.get(field) == value:
            field_hits += 1
        else:
            missed_fields.append((ds[i]['id'], field, value, got.get(field)))
print(f'field-level accuracy: {field_hits}/{field_total} = {field_hits / max(field_total, 1):.3f}')
print('example misses:', missed_fields[:10])

In [ ]:
# OCR-noise robustness: re-score test rows re-corrupted with the same augmentation.
import sys, random
sys.path.insert(0, '/content/packsure/ml/scripts')
from augmentation import augment_sample
from split import stable_hash_split  # noqa: prove split reproducibility

rng = random.Random(0)
noisy_true, noisy_pred = [], []
for i in range(len(ds)):
    sample = {'id': ds[i]['id'], 'product_id': ds[i]['product_id'], 'text': ' '.join(ds[i]['tokens']), 'entities': []}
    # rebuild entities from gold labels first
    tokens = ds[i]['tokens']
    entities, current, ctype, cursor = [], None, None, 0
    text = sample['text']
    for token, label in zip(tokens, ds[i]['labels']):
        start = text.index(token, cursor); cursor = start + len(token)
        if label.startswith('B-'):
            current, ctype = start, label[2:]
            entities.append({'start': start, 'end': cursor, 'label': ctype})
        elif label.startswith('I-') and entities:
            entities[-1]['end'] = cursor
    sample['entities'] = entities
    out = augment_sample(sample, rng, copies=1)
    if not out:
        continue
    noisy = out[0]
    enc = tokenizer(noisy['text'].split(), is_split_into_words=True, truncation=True, max_length=192, return_tensors='pt')
    with torch.no_grad():
        logits = model(**{k: v for k, v in enc.items()}).logits
    ids = logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids(0)
    pred_row, seen = [], set()
    for j, w in enumerate(word_ids):
        if w is None or w in seen: continue
        seen.add(w); pred_row.append(label_list[ids[j]])
    # gold labels for the noisy text from adjusted spans
    from bio import char_span_to_token_labels
    noisy_tokens = char_span_to_token_labels(noisy['text'], noisy['entities'])
    noisy_true.append([t['label'] for t in noisy_tokens][:len(pred_row)] + ['O'] * max(0, len(pred_row) - len(noisy_tokens)))
    noisy_pred.append(pred_row + ['O'] * max(0, len(noisy_tokens) - len(pred_row)))
from seqeval.metrics import f1_score as sf1
print(f'F1 on OCR-noised test copies: {sf1(noisy_true, noisy_pred):.3f}  (compare with clean F1 above)')

In [ ]:
# Inference latency (p50/p95) on GPU and CPU — measure, never guess.
import time

def latency(device, n=50):
    model.to(device)
    enc = tokenizer('MRP Rs.120/- Net Qty 500G', return_tensors='pt', truncation=True).to(device)
    times = []
    with torch.no_grad():
        for _ in range(n):
            t0 = time.perf_counter()
            model(**enc)
            times.append((time.perf_counter() - t0) * 1000)
    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    return sorted(times)[n // 2], sorted(times)[int(n * 0.95)]

p50, p95 = latency('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU/CPU p50={p50:.1f}ms p95={p95:.1f}ms')

**Checkpoint:** record the measured numbers in your report/README — and only these.
If the model is not good enough yet: annotate more data (notebook 02), retrain (03), re-evaluate (04).